# Download hourly Wikipedia event data

Download the 24 hourly Parquet files from `s3://dsan6000-wikipedia/hourly_parquet/` into the local `data/` subfolder. Run this notebook from the repository folder using `hw02-kernel`.

The source bucket is public, so unsigned requests allow downloads without AWS credentials.


In [1]:
from pathlib import Path

import boto3
from botocore import UNSIGNED
from botocore.config import Config

bucket = "dsan6000-wikipedia"
prefix = "hourly_parquet/"
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))


In [2]:
# A paginator includes every result even if the bucket listing spans multiple pages.
paginator = s3.get_paginator("list_objects_v2")
parquet_objects = [
    obj
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix)
    for obj in page.get("Contents", [])
    if obj["Key"].endswith(".parquet")
]
parquet_objects.sort(key=lambda obj: obj["Key"])

if not parquet_objects:
    raise RuntimeError(f"No Parquet files found in s3://{bucket}/{prefix}")

print(f"Found {len(parquet_objects)} hourly Parquet files.")
print(f"Total size: {sum(obj['Size'] for obj in parquet_objects) / 1_000_000:.2f} MB")


Found 24 hourly Parquet files.
Total size: 32.01 MB


In [ ]:
for obj in parquet_objects:
    key = obj["Key"]
    local_path = data_dir / Path(key).name
    s3.download_file(bucket, key, str(local_path))
    if local_path.stat().st_size != obj["Size"]:
        raise RuntimeError(f"Downloaded file size does not match S3: {local_path}")
    print(f"Downloaded {local_path}")

print(f"Finished: {len(parquet_objects)} files saved to {data_dir.resolve()}")


Downloaded data/20260901_040000.parquet
Downloaded data/20260901_050000.parquet
Downloaded data/20260901_060000.parquet
Downloaded data/20260901_070000.parquet
Downloaded data/20260901_080000.parquet
Downloaded data/20260901_090000.parquet
Downloaded data/20260901_100000.parquet
Downloaded data/20260901_110000.parquet
Downloaded data/20260901_120000.parquet
Downloaded data/20260901_130000.parquet
Downloaded data/20260901_140000.parquet
Downloaded data/20260901_150000.parquet
Downloaded data/20260901_160000.parquet
Downloaded data/20260901_170000.parquet
Downloaded data/20260901_180000.parquet
Downloaded data/20260901_190000.parquet
Downloaded data/20260901_200000.parquet
Downloaded data/20260901_210000.parquet
Downloaded data/20260901_220000.parquet
Downloaded data/20260901_230000.parquet
Downloaded data/20260902_000000.parquet
Downloaded data/20260902_010000.parquet
Downloaded data/20260902_020000.parquet
Downloaded data/20260902_030000.parquet
Finished: 24 files saved to /home/ubuntu